# NegotiateEnv Training - OpenEnv Hackathon

**Team**: Mayuka Reddy & Kushal Adhyaru  
**Environment**: https://kushaladhyaru-negotiate-env.hf.space  
**Repository**: https://github.com/MadhaviSG/openEnv-negotiateEnv

---

## What This Notebook Does

1. ✅ Clones your GitHub repository
2. ✅ Connects to your deployed HF Spaces environment
3. ✅ Runs baseline evaluation (before training)
4. ✅ Trains LLM agent using TRL GRPO (500 episodes)
5. ✅ Plots reward curves
6. ✅ Saves model to HuggingFace Hub

---

## GPU Requirements

- **Colab Pro (A100)**: TRL GRPO training (~1 hour) ✅ RECOMMENDED
- **Free Colab (T4)**: Unsloth 4-bit LoRA training (~2-3 hours)

**Select GPU**: Runtime → Change runtime type → A100

## 1. Check GPU

In [ ]:
# Check GPU type
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Clone Repository

In [ ]:
# Clone your GitHub repository
!git clone https://github.com/MadhaviSG/openEnv-negotiateEnv.git
%cd openEnv-negotiateEnv

# Verify files
!ls -la train_negotiate.py

## 3. Install Dependencies

In [ ]:
# Install the negotiate_env package
!pip install -e .

# Install basic dependencies
!pip install requests matplotlib numpy

## 4. Test Connection to Environment

In [ ]:
# Your deployed environment URL
ENV_URL = "https://kushaladhyaru-negotiate-env.hf.space"

print(f"Environment URL: {ENV_URL}")
print("Testing connection...\n")

import requests
import json

# Test health endpoint
try:
    response = requests.get(f"{ENV_URL}/health", timeout=10)
    if response.status_code == 200:
        print("✅ Environment server is accessible!")
        print(f"   Health check: {response.json()}")
    else:
        print(f"❌ Server returned status code: {response.status_code}")
except Exception as e:
    print(f"❌ Failed to connect: {e}")
    print("\nMake sure your Space is running:")
    print("https://huggingface.co/spaces/KushalAdhyaru/negotiate-env")

# Test reset endpoint
try:
    response = requests.post(f"{ENV_URL}/reset", json={}, timeout=10)
    if response.status_code == 200:
        obs = response.json()
        print(f"\n✅ Environment reset successful!")
        print(f"   Scenario: {obs.get('context', '')[:80]}...")
        print(f"   Max turns: {obs.get('max_turns')}")
        print(f"   Your budget: ${obs.get('your_max_price')}")
    else:
        print(f"\n⚠️  Reset returned status code: {response.status_code}")
except Exception as e:
    print(f"\n⚠️  Reset test failed: {e}")

## 5. Run Baseline Evaluation (Before Training)

In [ ]:
# Evaluate rule-based baseline
print("Running rule-based baseline (50 episodes)...\n")
!python baseline_rule.py --episodes 50 --env-url {ENV_URL}

In [ ]:
# Full evaluation with metrics
print("Running full evaluation (100 episodes)...\n")
!python evaluate.py --agent rule --episodes 100 --env-url {ENV_URL}

## 6. Detect GPU and Recommend Training Method

In [ ]:
import subprocess

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()

print(f"Detected GPU: {gpu_name}")
print("="*60)

if "A100" in gpu_name or "V100" in gpu_name:
    print("\n✅ RECOMMENDED: TRL GRPO Training")
    print("   - Faster training (~1 hour)")
    print("   - Better performance")
    print("   - Run the 'TRL GRPO Training' section below")
    training_method = "TRL"
elif "T4" in gpu_name:
    print("\n✅ RECOMMENDED: Unsloth 4-bit LoRA Training")
    print("   - Memory efficient")
    print("   - Works on free Colab")
    print("   - Run the 'Unsloth Training' section below")
    training_method = "Unsloth"
else:
    print(f"\n⚠️  Unknown GPU: {gpu_name}")
    print("   Try Unsloth training (more memory efficient)")
    training_method = "Unsloth"

print("\n" + "="*60)

## 7. TRL GRPO Training (A100/V100)

**Use this section if you have A100 or V100 GPU**

Training configuration:
- Model: Qwen/Qwen2.5-1.5B-Instruct
- Episodes: 500 (samples from 200 scenarios)
- Time: ~1 hour on A100
- Expected final reward: 0.60-0.65

In [ ]:
# Install TRL training dependencies
print("Installing TRL dependencies...\n")
!pip install -q trl>=0.29.0 transformers>=4.50.0 accelerate>=0.34.0 peft>=0.13.0
!pip install -q vllm>=0.6.0 torch>=2.5.0
print("\n✅ Dependencies installed!")

In [ ]:
# Run TRL GRPO training
print("Starting TRL GRPO training...")
print("This will take approximately 1 hour on A100.\n")
print("="*60)

!python train_negotiate.py \
    --vllm-mode colocate \
    --model-id Qwen/Qwen2.5-1.5B-Instruct \
    --output-dir negotiate-grpo-output \
    --num-episodes 500 \
    --max-turns 10 \
    --env-url {ENV_URL}

print("\n" + "="*60)
print("✅ Training complete!")

## 8. Unsloth 4-bit LoRA Training (T4)

**Use this section if you have T4 GPU (free Colab)**

Training configuration:
- Model: Qwen/Qwen2.5-1.5B-Instruct (4-bit quantized)
- Episodes: 300
- Time: ~2-3 hours on T4
- Expected final reward: 0.55-0.60

In [ ]:
# Install Unsloth (optimized for Colab)
print("Installing Unsloth dependencies...\n")
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl>=0.29.0 datasets requests
print("\n✅ Dependencies installed!")

In [ ]:
# Run Unsloth training
print("Starting Unsloth 4-bit LoRA training...")
print("This will take approximately 2-3 hours on T4.\n")
print("="*60)

!python train_negotiate_unsloth.py \
    --env-url {ENV_URL} \
    --model-id Qwen/Qwen2.5-1.5B-Instruct \
    --output-dir negotiate-unsloth-output \
    --num-episodes 300 \
    --max-turns 6

print("\n" + "="*60)
print("✅ Training complete!")

## 9. Plot Training Results

In [ ]:
# Plot reward curve
import os
from IPython.display import Image, display

print("Generating reward curve...\n")

# Determine which output directory exists
if os.path.exists("negotiate-grpo-output/trainer_state.json"):
    !python plot_reward_curve.py \
        --log-file negotiate-grpo-output/trainer_state.json \
        --out reward_curve.png
    training_dir = "negotiate-grpo-output"
elif os.path.exists("negotiate-unsloth-output/trainer_state.json"):
    !python plot_reward_curve.py \
        --log-file negotiate-unsloth-output/trainer_state.json \
        --out reward_curve.png
    training_dir = "negotiate-unsloth-output"
else:
    print("⚠️  No training log found. Make sure training completed successfully.")
    training_dir = None

# Display reward curve
if os.path.exists("reward_curve.png"):
    print("\n✅ Reward curve generated!\n")
    display(Image("reward_curve.png"))
else:
    print("⚠️  Reward curve not generated")

## 10. Run Demo with Trained Model

In [ ]:
# Run demo with rule-based agent
print("Running demo negotiation...\n")
print("="*60)
!python demo.py --difficulty medium --env-url {ENV_URL}
print("="*60)

## 11. Save Model to HuggingFace Hub

In [ ]:
# Install huggingface_hub
!pip install -q huggingface_hub

In [ ]:
# Login to HuggingFace
from huggingface_hub import notebook_login

print("Please login to HuggingFace to upload your model.")
print("Get your token from: https://huggingface.co/settings/tokens\n")
notebook_login()

In [ ]:
# Push model to HuggingFace Hub
from huggingface_hub import HfApi
import os

api = HfApi()

# Determine which output directory to upload
if os.path.exists("negotiate-grpo-output"):
    output_dir = "negotiate-grpo-output"
    model_name = "negotiate-env-qwen-grpo"
elif os.path.exists("negotiate-unsloth-output"):
    output_dir = "negotiate-unsloth-output"
    model_name = "negotiate-env-qwen-unsloth"
else:
    raise ValueError("No trained model found")

# Your HuggingFace username
repo_id = f"KushalAdhyaru/{model_name}"

print(f"Uploading {output_dir} to {repo_id}...\n")
print("This may take a few minutes...\n")

try:
    api.upload_folder(
        folder_path=output_dir,
        repo_id=repo_id,
        repo_type="model",
    )
    print(f"\n✅ Model uploaded successfully!")
    print(f"\nView your model at: https://huggingface.co/{repo_id}")
except Exception as e:
    print(f"\n❌ Upload failed: {e}")
    print("\nYou can manually upload the model later from the files.")

## 12. Download Results

In [ ]:
# Create a zip file with all results
print("Creating results archive...\n")

!zip -r training_results.zip \
    reward_curve.png \
    negotiate-*-output/ \
    *.log 2>/dev/null || true

print("\n✅ Results saved to training_results.zip")
print("\nDownload this file from Colab's file browser (left sidebar)")
print("Or run: from google.colab import files; files.download('training_results.zip')")

## 13. Training Summary

In [ ]:
# Display training summary
import json
import os

print("="*60)
print("TRAINING SUMMARY")
print("="*60)

# Find trainer state
trainer_state_path = None
if os.path.exists("negotiate-grpo-output/trainer_state.json"):
    trainer_state_path = "negotiate-grpo-output/trainer_state.json"
    method = "TRL GRPO"
elif os.path.exists("negotiate-unsloth-output/trainer_state.json"):
    trainer_state_path = "negotiate-unsloth-output/trainer_state.json"
    method = "Unsloth 4-bit LoRA"

if trainer_state_path and os.path.exists(trainer_state_path):
    with open(trainer_state_path) as f:
        state = json.load(f)
    
    log_history = state.get("log_history", [])
    
    # Extract rewards
    rewards = []
    for entry in log_history:
        reward = entry.get("env_reward") or entry.get("reward") or entry.get("train/env_reward")
        if reward is not None:
            rewards.append(float(reward))
    
    if rewards:
        print(f"\nTraining Method: {method}")
        print(f"Total Episodes: {len(rewards)}")
        print(f"\nInitial Reward: {rewards[0]:.4f}")
        print(f"Final Reward: {rewards[-1]:.4f}")
        print(f"Best Reward: {max(rewards):.4f}")
        print(f"Average Reward: {sum(rewards)/len(rewards):.4f}")
        print(f"\nImprovement: {(rewards[-1] - rewards[0]):.4f} ({((rewards[-1] - rewards[0])/rewards[0]*100):.1f}%)")
    else:
        print("\n⚠️  No reward data found in training logs")
else:
    print("\n⚠️  Training state not found")

print("\n" + "="*60)
print("\n✅ Training Complete!")
print("\nNext steps:")
print("1. Download training_results.zip")
print("2. Check your model on HuggingFace")
print("3. Submit to hackathon portal")
print("\n" + "="*60)

## 📋 Hackathon Submission Checklist

- ✅ Environment deployed to HF Spaces: https://huggingface.co/spaces/KushalAdhyaru/negotiate-env
- ✅ Dataset public: https://huggingface.co/datasets/mayukareddy/SyntheticSaasDataset
- ✅ Model trained and uploaded to HF Hub
- ✅ Code on GitHub: https://github.com/MadhaviSG/openEnv-negotiateEnv
- ⏳ Create demo video/screenshots
- ⏳ Submit to hackathon portal

---

## 🎯 Your URLs

- **Environment**: https://kushaladhyaru-negotiate-env.hf.space
- **GitHub**: https://github.com/MadhaviSG/openEnv-negotiateEnv
- **Model**: https://huggingface.co/KushalAdhyaru/negotiate-env-qwen-grpo
- **Dataset**: https://huggingface.co/datasets/mayukareddy/SyntheticSaasDataset

---

**Good luck with your hackathon submission! 🚀**